In [224]:
# Импортиране на библиотеките
import torch
from torch_geometric.data import HeteroData
from torch_geometric.nn import to_hetero
import torch.nn.functional as F

# Създаване на графа
data = HeteroData()

In [225]:
# Дефиниране на различните типове възли и добавяне на характеристики на възлите
data["author"].x = torch.randn(5,16)
data["paper"].x = torch.randn(8,16)
data["conference"].x = torch.randn(3,16)

In [230]:
# Дефиниране на различните типове ребра
# Автор → публикация
data["author","writes","paper"].edge_index = torch.tensor([
    [0,0,1,2,3,4],
    [0,1,2,3,4,5]])

# Публикация → конференция
data["paper","published_in","conference"].edge_index = torch.tensor([
    [0,1,2,3,4,5,6,7],
    [0,0,1,1,2,2,2,1]])

In [ ]:
from torch_geometric.transforms import ToUndirected
data = ToUndirected()(data)

In [231]:
# Преглед на графа
print(data)

HeteroData(
  author={ x=[5, 16] },
  paper={ x=[8, 16] },
  conference={ x=[3, 16] },
  (author, writes, paper)={ edge_index=[2, 6] },
  (paper, published_in, conference)={ edge_index=[2, 8] },
  (paper, rev_writes, author)={ edge_index=[2, 6] },
  (conference, rev_published_in, paper)={ edge_index=[2, 8] }
)


In [232]:
# Създаване на модел
from torch_geometric.nn import SAGEConv

class GNN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = SAGEConv((-1,-1),32)

        self.conv2 = SAGEConv((32,32),16)

    def forward(self,x,edge_index):

        x = self.conv1(x,edge_index)
        x = F.relu(x)
        x = self.conv2(x,edge_index)

        return x

In [233]:
# Моделът автоматично се преобразува в хетерогенен
model = GNN()

model = to_hetero(
    model,
    data.metadata())

In [234]:
# Изчисляване на представянията
out = model(
    data.x_dict,
    data.edge_index_dict)

In [240]:
# Извеждане на типовете възли
print(data.node_types)
# Извеждане на типовете ребра
print(data.edge_types)

['author', 'paper', 'conference']
[('author', 'writes', 'paper'), ('paper', 'published_in', 'conference'), ('paper', 'rev_writes', 'author'), ('conference', 'rev_published_in', 'paper')]


In [241]:
# Обучение на хетерогенната графова невронна мрежа
# Добавяне на етикети
data["paper"].y = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1])
# Създаване на обучаваща извадка
data["paper"].train_mask = torch.tensor(
    [True, True, True, True,
     False, False, False, False])
# Дефиниране на оптимизатор
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
# Функция на загубите
criterion = torch.nn.CrossEntropyLoss()
# Обучение на модела
for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    loss = criterion(
        out["paper"][data["paper"].train_mask],
        data["paper"].y[data["paper"].train_mask]  )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}, Loss = {loss.item():.4f}")

Epoch  10, Loss = 0.0862
Epoch  20, Loss = 0.0036
Epoch  30, Loss = 0.0004
Epoch  40, Loss = 0.0001
Epoch  50, Loss = 0.0001
Epoch  60, Loss = 0.0001
Epoch  70, Loss = 0.0001
Epoch  80, Loss = 0.0001
Epoch  90, Loss = 0.0001
Epoch 100, Loss = 0.0001


In [242]:
# Оценяване на модела
# Добавяне на тестова извадка
data["paper"].test_mask = torch.tensor(
    [False, False, False, False,
     True, True, True, True])
# Изчисляване на прогнозите
model.eval()
with torch.no_grad():
    out = model(data.x_dict, data.edge_index_dict)
    pred = out["paper"].argmax(dim=1)
# Изчисляване на мерките
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

y_true = data["paper"].y[data["paper"].test_mask].cpu()
y_pred = pred[data["paper"].test_mask].cpu()
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")


Accuracy : 0.7500
Macro F1 : 0.7333
